## LLM을 프로그래밍 방식으로 다루기

여러분은 아마도 ChatGPT와 같은 대규모 언어 모델(LLM)을 이전에 사용해본 경험이 있을 것입니다. 일반적으로는 사용자 인터페이스(UI)나 애플리케이션을 통해 상호작용하게 됩니다.

이 노트북에서는 Python을 사용하여 LLM에 직접 API를 통해 연결하고 질의하는 방법을 실습합니다. 이번 실습에서는 **Granite-7B-Instruct** 모델을 사용합니다. (https://huggingface.co/ibm-granite/granite-7b-instruct)  
이 모델은 IBM Research에서 개발한 완전한 오픈소스 모델이며, Apache 2.0 라이선스를 따릅니다.

해당 모델은 이미 실습용 클러스터에 배포되어 있습니다. 이 모델은 비교적 작은 편이지만, 실행을 위해 여전히 RAM 24GB를 갖춘 GPU가 필요하기 때문입니다...

### 요구 사항 및 라이브러리 임포트

실습 지침에 따라 올바른 워크벤치 이미지를 선택하여 실행하였다면, 필요한 모든 라이브러리가 이미 설치되어 있을 것입니다.  
(그렇지 않은 경우에는 다음 셀의 첫 번째 줄 주석을 해제하여 필요한 패키지를 설치하세요.)
그 후, 필요한 라이브러리들을 임포트 해보겠습니다.

In [ ]:
# 아래 줄은 올바른 워크벤치 이미지를 선택하지 않았거나, 이 노트북을 워크숍 환경 외부에서 사용하는 경우에만 주석을 해제하십시오.
# !pip install --no-cache-dir --no-dependencies --disable-pip-version-check -r requirements.txt

from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain_community.llms import VLLMOpenAI

### Langchain

Langchain (https://www.langchain.com/)은 언어 모델을 기반으로 하는 애플리케이션을 개발하기 위한 프레임워크입니다.  
LLM API를 적절하게 질의하기 위해 직접 작성해야 하는 반복적인 코드(boilerplate)를 Langchain이 대신 처리해 줍니다.

우리는 먼저 **llm** 인스턴스를 생성할 것입니다. 이 인스턴스는 LLM API를 호출할 수 있는 위치와 모델에 적용할 몇 가지 파라미터로 정의됩니다.  
예를 들어, `max_new_tokens`는 모델이 최대 512개의 토큰(단어나 단어의 일부)까지만 응답하도록 지시합니다.  
`temperature`는 여기서 아주 낮게 설정되어 있는데, 이렇게 설정하면 모델이 사실 기반에 충실하게 응답하고, 지나치게 "창의적"이지 않도록 만들 수 있습니다. 
이 실습에서는 멋진 시를 쓰는 것이 목적이 아니니까요!

In [ ]:
# LLM Inference Server URL
inference_server_url = "http://granite-7b-instruct-predictor.ic-shared-llm.svc.cluster.local:8080"

# LLM definition
llm = VLLMOpenAI(           # 우리는 vLLM OpenAI 호환 API 클라이언트를 사용하고 있습니다. 하지만 모델은 OpenAI가 아니라 OpenShift AI에서 실행되고 있습니다.
    openai_api_key="EMPTY",   # 따라서 OpenAI 키가 필요하지 않습니다.
    openai_api_base= f"{inference_server_url}/v1",
    model_name="granite-7b-instruct",
    top_p=0.92,
    temperature=0.01,
    max_tokens=512,
    presence_penalty=1.03,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

모델에 요청을 보낼 때마다 사용할 **템플릿**(즉, "프롬프트")도 필요합니다.

모델에 질의할 때는, 사용자가 입력한 내용을 그대로 보내는 경우는 거의 없습니다.  
그 입력 위에, 모델이 어떻게 응답해야 하는지를 알려주는 명확한 지침을 함께 전달해야 합니다.  
예를 들어, 무엇을 어떻게 대답해야 하는지, 어떤 내용은 대답하지 말아야 하는지, 어떤 말투로 응답해야 하는지 등을 지정해주어야 합니다.
아래 셸에 영어로 작성된 프롬프트는 다음과 같이 모델에게 지침을 내리고 있습니다.

* 당신은 도움이 되고, 정중하며, 정직한 어시스턴트입니다. 항상 가능한 한 도움이 되도록 하되, 안전을 최우선으로 해야 합니다.
* 당신은 어떤 질문을 받게 되며, 이에 대해 반드시 답변을 제공해야 합니다.
* 당신의 답변에는 해롭거나, 비윤리적이거나, 인종차별적이거나, 성차별적이거나, 독성 있거나, 위험하거나, 불법적인 내용이 포함되어서는 안 됩니다.
* 답변은 반드시 사회적으로 편향되지 않고, 긍정적인 성격을 유지해야 합니다.
* 질문이 말이 되지 않거나 사실적으로 일관되지 않는 경우에는, 틀린 답을 하는 대신 그 이유를 설명해야 합니다.
* 어떤 질문에 대해 답을 모를 경우에는 "모르겠습니다"라고 대답해야 합니다.

In [ ]:
template="""<|system|>
You are a helpful, respectful and honest assistant. Always be as helpful as possible, while being safe.
You will be asked a question, to which you must give an answer.
Your answer should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content.
Please ensure that your responses are socially unbiased and positive in nature.
If a question does not make any sense, or is not factually coherent, explain why instead of answering something not correct.
If you don't know the answer to a question, answer "I don't know".

<|user|>
### QUESTION:
{input}

### ANSWER:
<|assistant|>
"""
prompt = PromptTemplate(input_variables=["input"], template=template)

Langchain을 사용하면 이제 이러한 요소들을 손쉽게 "결합"하여, 모델에 질의할 때 사용할 **conversation** 객체를 만들 수 있습니다.

In [ ]:
conversation = prompt | llm

이제 모델에 질의할 준비가 되었습니다!

In [ ]:
query = "What is Artificial Intelligence?"

conversation.invoke(input=query); # ";" at the end of the line hides final output (repetition of the streamed answer)

실습을 계속 진행하시다가, 실습의 3.7 챕터에서 이 노트북으로 다시 돌아와서 선택적 실습을 더 진행하실 수도 있습니다.